# 9-RLAIF（GRPO / CISPO）

对应 `trainer/train_grpo.py`（PPO 见 `train_ppo.py`）。README 把一类 PO 算法写成：

$$\mathcal{J} = \mathbb{E}\big[f(r_t)\cdot g(A_t) - h(\mathrm{KL}_t)\big]$$

- **策略项** $r_t=\pi_\theta/\pi_{\mathrm{old}}$
- **优势项** $A_t$：GRPO 用同组生成的相对奖励，不需要 value model
- **正则项**：相对 ref 的 KL

GRPO：组内标准化 advantage + PPO clip。CISPO：把 ratio 上截断后当权重，乘到 token logprob 上。


In [ ]:
import os, sys, math, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append(os.path.abspath(".."))
import torch
from torch import optim
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from dataset.lm_dataset import PretrainDataset, SFTDataset, DPODataset, RLAIFDataset, AgentRLDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("../model")
print("device:", device, "vocab:", tokenizer.vocab_size)
def tiny_model(use_moe=False):
    cfg = MiniMindConfig(hidden_size=64, num_hidden_layers=1, use_moe=use_moe)
    return MiniMindForCausalLM(cfg).to(device), cfg

import torch.nn.functional as F


## 组相对优势（不依赖完整 rollout）

同一 prompt 生成 G 条回复，用组内 mean/std 标准化 reward。这是 GRPO 相对 PPO 最关键的简化。


In [ ]:
rewards = torch.tensor([1.2, 0.4, -0.2, 0.8]).view(2, 2)  # batch=2, G=2
mean = rewards.mean(1, keepdim=True)
std = rewards.std(1, unbiased=False, keepdim=True)
adv = (rewards - mean) / (std + 1e-4)
print("rewards\n", rewards)
print("advantages\n", adv)


## 极简 GRPO / CISPO loss

真实脚本会走 `rollout_engine` 采样、算 per-token logps、再 clip。这里用随机 tensor 把公式跑通。


In [ ]:
B, T, G = 2, 6, 2
old_logp = torch.randn(B * G, T)
new_logp = old_logp + 0.05 * torch.randn(B * G, T)
ref_logp = old_logp - 0.02 * torch.randn(B * G, T)
mask = torch.ones(B * G, T)
advantages = adv.reshape(-1, 1)

ratio = torch.exp(new_logp - old_logp)
kl = torch.exp(ref_logp - new_logp) - (ref_logp - new_logp) - 1
eps, beta = 0.2, 0.04

clipped = torch.clamp(ratio, 1 - eps, 1 + eps)
grpo = -((torch.min(ratio * advantages, clipped * advantages) - beta * kl) * mask).sum(1) / mask.sum(1)
print("grpo", float(grpo.mean()))

cispo_w = torch.clamp(ratio, max=1 + eps).detach()
cispo = -((cispo_w * advantages * new_logp - beta * kl) * mask).sum(1) / mask.sum(1)
print("cispo", float(cispo.mean()))


## 看 RLAIF prompt

真实训练：`cd trainer && python train_grpo.py`，数据 `../dataset/rlaif.jsonl`。需要 reward model + rollout。PPO 则还要 value head，见 `train_ppo.py`。


In [ ]:
ds = RLAIFDataset("./toydata/rlaif_data.jsonl", tokenizer, thinking_ratio=0.5)
print(ds[0]["prompt"])
